In [7]:
import os
import chromadb

from docx import Document as DocxDocument

from llama_index.core import VectorStoreIndex
from llama_index.core import Settings
from llama_index.core.schema import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.groq import Groq

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core import StorageContext

In [ ]:
def parse_docx(file_path):

    doc = DocxDocument(file_path)
    docs = []

    file_name = os.path.basename(file_path)

    # xác định ngành
    if "CNTT" in file_name:
        program = "CNTT"

    elif "NHTTTK" in file_name:
        program = "NHTTTK"

    elif "KHMT" in file_name:
        program = "KHMT"

    elif "QuyetDinhChung" in file_name:
        program = "QD_CHUNG"

    else:
        program = "UNKNOWN"

    # xác định khóa K19 K20...
    match = re.search(r'K(\d+)', file_name)

    if match:
        khoa = f"K{match.group(1)}"
    else:
        khoa = "UNKNOWN"

    # đọc paragraph
    for i, p in enumerate(doc.paragraphs):

        text = p.text.strip()

        if text:

            docs.append(
                Document(
                    text=text,
                    metadata={
                        "source": file_name,
                        "program": program,
                        "khoa": khoa,
                        "type": "paragraph",
                        "paragraph_id": i
                    }
                )
            )

    # đọc bảng
    for table in doc.tables:

        for r, row in enumerate(table.rows):

            row_text = " | ".join(cell.text.strip() for cell in row.cells)

            if row_text:

                docs.append(
                    Document(
                        text=row_text,
                        metadata={
                            "source": file_name,
                            "program": program,
                            "khoa": khoa,
                            "type": "table",
                            "row_id": r
                        }
                    )
                )

    return docs

In [3]:
data_dir = r"D:/NCKH/agent/NCKH/data_raw"

documents = []

for f in os.listdir(data_dir):

    if f.endswith(".docx") and not f.startswith("~$"):

        file_path = os.path.join(data_dir, f)

        documents.extend(parse_docx(file_path))

print("Total documents:", len(documents))

Total documents: 794


In [5]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3"
)

Settings.embed_model = embed_model

In [4]:
splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=50
)

nodes = splitter.get_nodes_from_documents(documents)

print("Total nodes:", len(nodes))

Total nodes: 794


In [8]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    "ctdt_index"
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

In [24]:
import os
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

# 1. Cấu hình Model (Đảm bảo đồng bộ với các bước trước)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
Settings.llm = Groq(model="llama-3.1-8b-instant", api_key="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")

# 2. Thiết lập đường dẫn lưu trữ
db_path = r"D:\NCKH\agent\NCKH\vector_store"
if not os.path.exists(db_path):
    os.makedirs(db_path)

# 3. Khởi tạo ChromaDB Client với chế độ lưu trữ vĩnh viễn (Persistent)
db = chromadb.PersistentClient(path=db_path)

# 4. Tạo hoặc lấy Collection (giống như bảng trong SQL)
chroma_collection = db.get_or_create_collection("dnu_collection")

# 5. Cấu hình Vector Store cho LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 6. Tạo Index và LƯU vào ổ đĩa
# Giả sử 'documents' là danh sách tài liệu bạn đã load
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context,
    show_progress=True
)

print(f"Đã lưu vector database thành công vào: {db_path}")

Generating embeddings: 100%|██████████| 794/794 [02:43<00:00,  4.87it/s]


Đã lưu vector database thành công vào: D:\NCKH\agent\NCKH\vector_store


In [9]:
os.environ["GROQ_API_KEY"] = "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"


In [10]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

llm = Groq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)

# gán vào Settings của LlamaIndex
Settings.llm = llm

In [11]:
print(Settings.llm)

callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x000002758E93E090> system_prompt=None messages_to_prompt=<function messages_to_prompt at 0x00000275D0B94540> completion_to_prompt=<function default_completion_to_prompt at 0x00000275D0EFAFC0> output_parser=None pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'> query_wrapper_prompt=None model='llama-3.1-8b-instant' temperature=0.1 max_tokens=None logprobs=None top_logprobs=0 additional_kwargs={} max_retries=3 timeout=60.0 default_headers=None reuse_client=True api_key='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' api_base='https://api.groq.com/openai/v1' api_version='' strict=False reasoning_effort=None modalities=None audio_config=None context_window=3900 is_chat_model=True is_function_calling_model=True should_use_structured_outputs=False tokenizer=None


In [12]:
print(Settings.embed_model)

model_name='BAAI/bge-m3' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x000002758E93E090> num_workers=None embeddings_cache=None max_length=8192 normalize=True query_instruction=None text_instruction=None cache_folder=None show_progress_bar=False


In [15]:
query_engine = index.as_query_engine(
    similarity_top_k=20
)

In [16]:

response = query_engine.query("Trí tuệ nhân tạo có số tín chỉ là bao nhiêu, học vào kỳ nào, mã môn là mấy? ")
print(response)

FIT4011 có số tín chỉ là 3, học vào học kỳ 5, mã môn là FIT4011.


In [18]:

response = query_engine.query("Quy trình rút bớt học phần gồm những bước nào, chi tiết từng bước ra sao? ")
print(response)


Quy trình rút bớt học phần được thực hiện theo các bước sau:

Bước 1: Sinh viên đăng nhập cổng thông tin sinh viên để đăng ký rút học phần.

Bước 2: Khoa/Viện/Trung tâm xác nhận cho sinh viên rút bớt học phần đã đăng ký.

Bước 3: Phòng Quản lý Đào tạo xác nhận và tiến hành hủy lớp cho sinh viên.

Bước 4: Phòng Tài chính kế toán tiến hành xác nhận và hủy tài chính/trả lại tài chính cho sinh viên (nếu có).


In [19]:

response = query_engine.query("Đối với sinh viên nghỉ học tạm thời, bảo lưu nhập học lại có quy trình, thủ tục thế nào")
print(response)

Để nhập học lại sau khi nghỉ học tạm thời, sinh viên cần thực hiện các bước sau:

- Trước khi bắt đầu học kỳ mới, sinh viên phải làm thủ tục theo quy định của Trường.
- Thời gian để làm thủ tục này là ít nhất 10 ngày làm việc.
- Sau khi hoàn thành thủ tục, sinh viên có thể tiếp tục học tập tại Trường.


In [ ]:
response = query_engine.query("bạn có thể viết 1 đoạn code C++  tính tổng 3 số nguyên dương không?")
print(response)

AttributeError: 'dict' object has no attribute 'condition'